In [ ]:
import numpy as np
import skrobot

In [ ]:
robot_fk_model = skrobot.models.urdf.RobotModelFromURDF(urdf_file="./urdf/two_link_fk.urdf")
robot_ik_model = skrobot.models.urdf.RobotModelFromURDF(urdf_file="./urdf/two_link_ik.urdf")

In [ ]:
viewer = skrobot.viewers.TrimeshSceneViewer(resolution=(640, 480))
viewer.add(robot_fk_model)
viewer.add(robot_ik_model)
viewer.show()

In [ ]:
viewer.set_camera(angles=[np.deg2rad(90), 0, np.deg2rad(180)])
viewer.redraw()

In [ ]:
viewer.add(robot_fk_model)
viewer.add(robot_ik_model)
viewer.redraw()

In [ ]:
viewer.delete(robot_fk_model)
viewer.delete(robot_ik_model)
viewer.redraw()

In [ ]:
fk_tip = skrobot.model.Box(extents=(0.01, 0.01, 0.01), face_colors=(0.75, 0.75, 0.75, 0.5))
fk_tip.translate((0.07071, 0, 0))
robot_fk_model.link2.assoc(fk_tip, relative_coords='local')

ik_tip = skrobot.model.Box(extents=(0.01, 0.01, 0.01), face_colors=(0.75, 0.75, 0.75, 0.5))
ik_tip.translate((0.05, 0, 0))
robot_ik_model.link2.assoc(ik_tip, relative_coords='local')
viewer.add(fk_tip)
viewer.add(ik_tip)

In [ ]:
viewer.delete(fk_tip)
viewer.delete(ik_tip)


In [ ]:
viewer.close()

In [ ]:
for link in robot_model.link_list:
    print(link.name)

In [ ]:
for joint in robot_model.joint_list:
    print(joint.name)

In [73]:
robot_fk_model.joint2.joint_angle(-np.pi*3/4)
viewer.redraw()

In [59]:
robot_ik_model.joint1.joint_angle(-np.pi*1/4)
robot_ik_model.joint2.joint_angle(-np.pi*3/4)
viewer.redraw()

In [ ]:
print(fk_tip.worldpos())
link_list = robot_ik_model.link_list[1:]
print(robot_ik_model.link_list)
#target_coords = fk_tip.copy_worldcoords()
target_coords = skrobot.coordinates.Coordinates(pos=fk_tip.worldpos())
print(target_coords)
ik_end_coords = skrobot.coordinates.CascadedCoords(
    parent=robot_ik_model.ee_link, 
    name="ik_end_coords")
move_target = ik_end_coords
success = robot_ik_model.inverse_kinematics(
                                    target_coords = target_coords, 
                                    move_target = move_target, 
                                    link_list=link_list, 
                                    translation_only=True, 
                                    thre=0.0001,           # 0.1mm position tolerance
                                    rthre=np.deg2rad(0.1)  # 0.1 degree rotation tolerance)
                                )

print(success)
print(robot_ik_model.angle_vector())
viewer.redraw()

In [70]:
def phi_acute(theta):
    return 2.0 * np.arctan(((3-2*np.sqrt(2)) * np.sin(theta)) / (1-np.cos(theta)))

In [71]:
print(np.rad2deg(phi_acute(np.deg2rad(45))))

44.99999999999996


In [78]:
theta = np.pi/16
robot_fk_model.joint2.joint_angle(theta - np.pi)
robot_ik_model.joint1.joint_angle(-phi_acute(theta))
robot_ik_model.joint2.joint_angle(theta - np.pi)
viewer.redraw()

In [ ]:
link_list[1:]

In [ ]:
# robots = [robot_fk_model, robot_ik_model]
robots = [fk_tip, ik_tip]
for robot in robots:
    axis = skrobot.model.Axis(
        axis_radius=0.001,
        axis_length=0.02,
        pos=robot.worldpos(),
        rot=robot.worldrot()
    )
    viewer.add(axis)